## Trying Other Models

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load data
X = np.load('Datasets/kryptonite-45-X.npy')
y = np.load('Datasets/kryptonite-45-y.npy')

# Convert X and y to DataFrame for easier inspection
# Assuming each column in X represents a feature
df = pd.DataFrame(X, columns=[f'Feature_{i}' for i in range(X.shape[1])])

df['Target'] = y  # Add target as a separate column
df

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.layers import Input
import tensorflow as tf
import matplotlib.pyplot as plt
from tf_fourier_features import FourierFeatureProjection

# Load data
X = np.load('Datasets/kryptonite-45-X.npy')
y = np.load('Datasets/kryptonite-45-y.npy')

# Normalise data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into train, validation, and test sets (70-15-15 split)
X_train, X_temp, y_train, y_temp = train_test_split(X_scaled, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Apply PLS for dimensionality reduction
n_components = 20  # Using 15 latent variables
pls = PLSRegression(n_components=n_components)
X_train_pls = pls.fit_transform(X_train, y_train)[0]
X_val_pls = pls.transform(X_val)
X_test_pls = pls.transform(X_test)

print("Training set after PLS:", X_train_pls.shape, y_train.shape)
print("Validation set after PLS:", X_val_pls.shape, y_val.shape)
print("Test set after PLS:", X_test_pls.shape, y_test.shape)

# Define a neural network with Fourier feature projection and dense layers for binary classification
ip = Input(shape=(n_components,))
# Fourier Feature Projection layer
x = FourierFeatureProjection(gaussian_projection=128, gaussian_scale=10)(ip)

# Neural network layers
x = Dense(2048, activation='relu')(x)
x = Dense(1024, activation='relu')(x)
x = Dense(512, activation='relu')(x)
x = Dense(256, activation='relu')(x)
x = Dense(128, activation='relu')(x)
x = Dense(32, activation='relu')(x)
x = Dense(16, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)  # Sigmoid for binary classification

# Define the model
model = tf.keras.Model(inputs=ip, outputs=output)

# Compile the model with SGD (Stochastic Gradient Descent) for better generalisation
model.compile(optimizer=SGD(learning_rate=0.0001, momentum=0.01), loss='binary_crossentropy', metrics=['accuracy'])

# Define early stopping callback
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model with early stopping
history = model.fit(
    X_train_pls, y_train,
    validation_data=(X_val_pls, y_val),
    epochs=30,
    batch_size=16,
    verbose=1,
    callbacks=[early_stopping]
)

# Plot the loss curves for training and validation
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Evaluate the model on the test set for loss and accuracy
test_loss, test_accuracy = model.evaluate(X_test_pls, y_test, verbose=1)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")